# 33B The contract: Pydantic, and the gate behind it
The model's output is an object with rules, not a string. The rules run twice: in the agent (no database needed) and in the server (the authority).

In [ ]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

In [ ]:
cases = ["SELECT bdc FROM holdings_v", "WITH x AS (SELECT * FROM holdings) SELECT * FROM x",
         "DELETE FROM holdings", "SELECT 1; DROP TABLE holdings", "SELECT * FROM sqlite_master",
         "SELECT * FROM holdings_v h JOIN secrets s ON 1"]
for s in cases:
    try:
        gate.static_check(s); print(f"  ok       {s}")
    except gate.GateError as e:
        print(f"  {e.code:22s} {s}")

## The server's gate compiles the SQL under an authorizer, on a read-only connection
Keyword filters miss things; SQLite does not. `pragma_table_info` and `load_extension` pass a keyword check and are still refused:

In [ ]:
con = gate.connect_ro(BASE)
for s in ["SELECT * FROM pragma_table_info('holdings')", "SELECT load_extension('x')",
          "WITH x AS (SELECT 1) UPDATE filings SET name = 'x'",
          "WITH RECURSIVE c(x) AS (SELECT 1 UNION ALL SELECT x + 1 FROM c) SELECT MAX(x) FROM c"]:
    try:
        gate.run_query(con, s, timeout_s=0.5); print("  ran     ", s)
    except gate.GateError as e:
        print(f"  {e.code:15s}", s[:70])

## The same rules as a Pydantic model
This cell runs only if `pydantic` is installed.

In [ ]:
import importlib.util
if importlib.util.find_spec("pydantic") is None:
    print("pydantic not installed: skipped (pip install -r requirements.txt)")
else:
    from typing import Optional
    from pydantic import BaseModel, ValidationError, field_validator, model_validator

    class SQLQuery(BaseModel):
        sql: Optional[str] = None
        decline: Optional[str] = None
        explanation: str = ""

        @field_validator("sql")
        @classmethod
        def gated(cls, v):
            return None if v is None else gate.static_check(v)

        @model_validator(mode="after")
        def exactly_one(self):
            if (self.sql is None) == (self.decline is None):
                raise ValueError("give exactly one of sql or decline")
            return self

    print(SQLQuery(sql="SELECT bdc FROM holdings_v;").sql)
    for bad in [{"sql": "DROP TABLE holdings"}, {"sql": None}, {"sql": "SELECT 1", "decline": "no"}]:
        try:
            SQLQuery(**bad)
        except ValidationError as e:
            print("  refused:", e.errors()[0]["msg"])

**Your turn (Part 1, 12 points).** In `agent/contract.py`:
1. `SchemaCatalog`, `QueryRequest`, `SQLQuery`, `QueryResult` as Pydantic v2 models.
2. `SQLQuery` validates with `gate.static_check` and enforces exactly one of `sql` or `decline`.
3. `validate_query(dict)` returns `(model, None)` or `(None, {"code": ..., "detail": ...})`, with the gate's code, so the repair edge can tell the model what went wrong.